In [ ]:
# ==========================================
# 1. THƯ VIỆN & CẤU HÌNH BAN ĐẦU
# ==========================================
import json
import logging
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    hamming_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


In [ ]:
# ==========================================
# 2. THIẾT LẬP CẤU HÌNH & THƯ MỤC CẤU TRÚC
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
COLOR_LOSS_WEIGHT = 2.0  # Tăng trọng số cho task Color để khắc phục lệch hội tụ

NUM_SHAPE_CLASSES = 16
NUM_COLOR_CLASSES = 12

# Tự động phát hiện đường dẫn Dataset trên Kaggle / Máy cục bộ
candidate_base_dirs = [
    Path("/kaggle/input/datasets/baoakabin/rximage-dataset/Data_rximage_kaggle/rximage"),
    Path("/kaggle/input/rximage-dataset/Data_rximage_kaggle/rximage"),
    Path("/kaggle/input/rximage_dataset/Data_rximage_kaggle/rximage"),
    Path("/kaggle/input/datasets/baoakabin/rximage-dataset/Data_rximage_kaggle"),
    Path("/kaggle/input/rximage-dataset/Data_rximage_kaggle"),
    Path("/kaggle/input/datasets/baoakabin/rximage-dataset"),
    Path("/kaggle/input/rximage-dataset/Data/rximage"),
    Path("/kaggle/input/rximage_dataset/Data/rximage"),
    Path("/kaggle/input/rximage-dataset"),
    Path("/kaggle/input/rximage_dataset"),
    Path("/kaggle/input/rximage_processed/Data/rximage"),
    Path("/kaggle/input/rximage-processed/Data/rximage"),
    Path("c:/ML_DL_Project/Data_rximage_kaggle/rximage"),
    Path("c:/ML_DL_Project/Data_rximage_kaggle"),
    Path("c:/ML_DL_Project/Data/rximage"),
    Path("Data/rximage")
]

BASE_DIR = None
for p in candidate_base_dirs:
    if p.exists():
        if (p / 'combined').exists() or len(list(p.glob('*.csv'))) > 0:
            BASE_DIR = p
            break

if BASE_DIR is None:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.rglob("combined"):
            if sub.is_dir():
                BASE_DIR = sub.parent
                break
        if BASE_DIR is None:
            for csv_file in kaggle_input.rglob("*.csv"):
                if "train" in csv_file.name.lower():
                    BASE_DIR = csv_file.parent.parent if csv_file.parent.name == "combined" else csv_file.parent
                    break

if BASE_DIR is None:
    BASE_DIR = Path("/kaggle/input/datasets/baoakabin/rximage-dataset/Data_rximage_kaggle/rximage")

COMBINED_DIR = BASE_DIR / "combined" if (BASE_DIR / "combined").exists() else BASE_DIR

# Ưu tiên nạp tập train gốc train_combined_crop.csv
if (COMBINED_DIR / "train_combined_crop.csv").exists():
    train_csv = COMBINED_DIR / "train_combined_crop.csv"
elif (COMBINED_DIR / "augmented_train_combined.csv").exists():
    train_csv = COMBINED_DIR / "augmented_train_combined.csv"
else:
    train_csv = COMBINED_DIR / "train_combined_crop.csv"

val_csv = COMBINED_DIR / "val_combined_crop.csv"
test_csv = COMBINED_DIR / "test_combined_crop.csv"

# Tìm thư mục chứa ảnh thông minh (Quét cả thư mục cha để tránh sót ảnh)
def find_image_directory(base_dir):
    search_roots = [base_dir]
    if base_dir.parent and base_dir.parent.exists():
        search_roots.append(base_dir.parent)
    if base_dir.parent and base_dir.parent.parent and base_dir.parent.parent.exists():
        search_roots.append(base_dir.parent.parent)
    
    for search_root in search_roots:
        for folder_name in ["image_all", "images", "image_all_crop", "images_all", "image_crop", "images_crop"]:
            p = search_root / folder_name
            if p.exists() and (len(list(p.glob("*.jpg"))) > 10 or len(list(p.glob("*.png"))) > 10):
                return p
        if len(list(search_root.glob("*.jpg"))) > 10 or len(list(search_root.glob("*.png"))) > 10:
            return search_root

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.rglob("*"):
            if sub.is_dir() and (len(list(sub.glob("*.jpg"))) > 10 or len(list(sub.glob("*.png"))) > 10):
                return sub
    return base_dir / "image_all"

IMG_DIR = find_image_directory(BASE_DIR)

# Checkpoint LastBlock (Layer 4) từ notebook trước (nếu có)
candidate_checkpoints = [
    Path("/kaggle/input/resnet-18-lastblock-head/outputs/lastblock/checkpoints/best_resnet18_lastblock_layer4.pth"),
    Path("/kaggle/input/resnet18-lastblock/best_resnet18_lastblock_layer4.pth"),
    Path("/kaggle/working/outputs/lastblock/checkpoints/best_resnet18_lastblock_layer4.pth")
]
LASTBLOCK_CHECKPOINT_PATH = None
for ckpt in candidate_checkpoints:
    if ckpt.exists():
        LASTBLOCK_CHECKPOINT_PATH = ckpt
        break

# Thư mục xuất kết quả riêng cho Heads Fine-tuning
OUTPUT_DIR = Path("/kaggle/working/outputs/heads_finetuned")
SUBDIRS = ["checkpoints", "logs", "metrics", "plots", "predictions"]

PATHS = {}
for sub in SUBDIRS:
    path = OUTPUT_DIR / sub
    path.mkdir(parents=True, exist_ok=True)
    PATHS[sub] = path

def setup_logger(name, log_file):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(log_file)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(message)s"))
    logger.addHandler(handler)
    return logger

logger_heads = setup_logger("heads_finetune", PATHS["logs"] / "training.log")

print(f"✓ Đã thiết lập thư mục output: {OUTPUT_DIR.resolve()}")
print(f"✓ Đường dẫn Dataset phát hiện được: {BASE_DIR.resolve() if BASE_DIR.exists() else BASE_DIR}")
print(f"✓ File CSV Train sử dụng: {train_csv.name if train_csv.exists() else train_csv}")
print(f"✓ Thư mục CSV Combined: {COMBINED_DIR.resolve() if COMBINED_DIR.exists() else COMBINED_DIR}")
print(f"✓ Thư mục Ảnh Image All: {IMG_DIR.resolve() if IMG_DIR.exists() else IMG_DIR}")
print(f"✓ Checkpoint LastBlock: {LASTBLOCK_CHECKPOINT_PATH}")
print(f"✓ Thiết bị tính toán: {DEVICE}")
print(f"✓ Trọng số Loss Color (Alpha): {COLOR_LOSS_WEIGHT}")

In [ ]:
# ==========================================
# 3. DATASET CLASS
# ==========================================
class RxImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, shape_encoder=None, mlb_color=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.path_cache = {}

        # 1. Tự động mã hóa nhãn Shape nếu thiếu cột 'shape_label'
        if "shape_label" in self.df.columns:
            self.shape_labels = self.df["shape_label"].values
        elif "shape" in self.df.columns:
            from sklearn.preprocessing import LabelEncoder
            if shape_encoder is None:
                self.shape_encoder = LabelEncoder()
                self.shape_labels = self.shape_encoder.fit_transform(self.df["shape"].fillna("UNKNOWN").astype(str))
            else:
                self.shape_encoder = shape_encoder
                self.shape_labels = self.shape_encoder.transform(self.df["shape"].fillna("UNKNOWN").astype(str))
        else:
            raise KeyError("Không tìm thấy cột 'shape_label' hoặc 'shape' trong tệp CSV.")

        # 2. Tự động mã hóa nhãn Color (Multi-label) nếu thiếu các cột 'color_'
        self.color_cols = [c for c in self.df.columns if c.startswith("color_")]
        if len(self.color_cols) > 0:
            self.color_labels = self.df[self.color_cols].values.astype(np.float32)
        elif "color" in self.df.columns:
            from sklearn.preprocessing import MultiLabelBinarizer
            def parse_colors(color_str):
                if not color_str or pd.isna(color_str):
                    return ["unknown"]
                colors = str(color_str).replace(";", " ").replace("/", " ").replace(",", " ").split()
                return [c.strip().lower() for c in colors if c.strip()]
            color_series = self.df["color"].apply(parse_colors)
            if mlb_color is None:
                self.mlb_color = MultiLabelBinarizer()
                color_bin = self.mlb_color.fit_transform(color_series)
            else:
                self.mlb_color = mlb_color
                color_bin = self.mlb_color.transform(color_series)
            self.color_labels = color_bin.astype(np.float32)
            self.color_cols = [f"color_{c}" for c in getattr(self.mlb_color, "classes_", [])]
        else:
            raise KeyError("Không tìm thấy các cột 'color_' hoặc cột 'color' trong tệp CSV.")

    def _resolve_img_path(self, filename):
        if filename in self.path_cache:
            return self.path_cache[filename]

        clean_filename = Path(filename).name
        candidates = [
            self.img_dir / filename,
            self.img_dir / clean_filename,
            self.img_dir.parent / filename,
            self.img_dir.parent / clean_filename,
            self.img_dir.parent / "image_all" / clean_filename,
            self.img_dir.parent.parent / "image_all" / clean_filename if self.img_dir.parent else None,
            self.img_dir.parent.parent / clean_filename if self.img_dir.parent else None,
            BASE_DIR / filename,
            BASE_DIR / clean_filename,
            BASE_DIR.parent / "image_all" / clean_filename if BASE_DIR else None,
        ]

        for cand in candidates:
            if cand and cand.exists():
                self.path_cache[filename] = cand
                return cand

        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            matches = list(kaggle_input.rglob(clean_filename))
            if matches:
                self.path_cache[filename] = matches[0]
                return matches[0]

        default_p = self.img_dir / filename
        self.path_cache[filename] = default_p
        return default_p

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row.get("rxnavImageFileName", row.get("filename", ""))).strip()
        img_path = self._resolve_img_path(filename)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)
        shape_target = torch.tensor(self.shape_labels[idx], dtype=torch.long)
        color_target = torch.tensor(self.color_labels[idx], dtype=torch.float32)
        return image, shape_target, color_target

In [ ]:
# ==========================================
# 4. TRANSFORMS & DATALOADERS (AUGMENTATION: FLIP + ROTATION, KHÔNG CÓ COLORJITTER)
# ==========================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}
train_csv = find_csv_file(COMBINED_DIR, "augmented_train_combined.csv", ["augmented", "train"])
val_csv = find_csv_file(COMBINED_DIR, "val_combined_crop.csv", ["val"])

print(f"✓ Sử dụng Train CSV: {train_csv}")
print(f"✓ Sử dụng Val CSV: {val_csv}")

train_dataset = RxImageDataset(train_csv, IMG_DIR, transform=data_transforms["train"])
shape_encoder = getattr(train_dataset, 'shape_encoder', None)
mlb_color = getattr(train_dataset, 'mlb_color', None)
val_dataset = RxImageDataset(val_csv, IMG_DIR, transform=data_transforms["val"], shape_encoder=shape_encoder, mlb_color=mlb_color)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
NUM_SHAPE_CLASSES = len(np.unique(train_dataset.shape_labels))
NUM_COLOR_CLASSES = len(train_dataset.color_cols)
print(f"✓ Số lớp Shape: {NUM_SHAPE_CLASSES} | Số nhãn Color: {NUM_COLOR_CLASSES}")

In [ ]:
# ==========================================
# 5. MÔ HÌNH: FREEZE BACKBONE + RE-DESIGN HEADS
# ==========================================
if 'NUM_SHAPE_CLASSES' not in globals():
    NUM_SHAPE_CLASSES = 16
if 'NUM_COLOR_CLASSES' not in globals():
    NUM_COLOR_CLASSES = 12
if 'LASTBLOCK_CHECKPOINT_PATH' not in globals():
    LASTBLOCK_CHECKPOINT_PATH = None
if 'DEVICE' not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MultiTaskResNet18_HeadsFinetune(nn.Module):
    def __init__(
        self,
        num_shape_classes=NUM_SHAPE_CLASSES,
        num_color_classes=NUM_COLOR_CLASSES,
        pretrained=True,
        lastblock_checkpoint=None,
    ):
        super(MultiTaskResNet18_HeadsFinetune, self).__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)

        # Nếu có checkpoint LastBlock cũ, nạp vào layer4 trước khi freeze
        if lastblock_checkpoint and os.path.exists(str(lastblock_checkpoint)):
            print(f"Loading lastblock weights from: {lastblock_checkpoint}")
            self.backbone.layer4.load_state_dict(torch.load(lastblock_checkpoint, map_location="cpu"))
        else:
            print("Dùng weights mặc định cho Backbone/Layer4.")

        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # FREEZE HOÀN TOÀN BACKBONE (Bao gồm cả Layer 4)
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 1. Shape Head (Giữ đơn giản với Dropout + Linear)
        self.fc_shape = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_shape_classes)
        )

        # 2. Re-designed Color Head (Thêm Linear + BatchNorm + ReLU để học đặc trưng tốt hơn)
        self.fc_color = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_color_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        shape_out = self.fc_shape(features)
        color_out = self.fc_color(features)
        return shape_out, color_out


In [ ]:
# ==========================================
# 6. KHỞI TẠO DATALOADERS & MÔ HÌNH
# ==========================================
if 'NUM_SHAPE_CLASSES' not in globals():
    NUM_SHAPE_CLASSES = 16
if 'NUM_COLOR_CLASSES' not in globals():
    NUM_COLOR_CLASSES = 12
if 'LASTBLOCK_CHECKPOINT_PATH' not in globals():
    LASTBLOCK_CHECKPOINT_PATH = None
if 'DEVICE' not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    train_dataset = RxImageDataset(train_csv, IMG_DIR, transform=data_transforms["train"])
    val_dataset = RxImageDataset(val_csv, IMG_DIR, transform=data_transforms["val"])

    NUM_SHAPE_CLASSES = len(np.unique(train_dataset.shape_labels))
    NUM_COLOR_CLASSES = len(train_dataset.color_cols)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    print(f"✓ Đã khởi tạo Dataloaders: Train ({len(train_dataset)} mẫu), Val ({len(val_dataset)} mẫu)")
    print(f"✓ Tự động xác định: {NUM_SHAPE_CLASSES} lớp Shape, {NUM_COLOR_CLASSES} lớp Color.")
except Exception as e:
    print(f"⚠️ Cảnh báo khởi tạo Dataset: {e}")

# Khởi tạo mô hình Multi-task ResNet-18
model = MultiTaskResNet18_HeadsFinetune(
    num_shape_classes=NUM_SHAPE_CLASSES,
    num_color_classes=NUM_COLOR_CLASSES,
    pretrained=True,
    lastblock_checkpoint=LASTBLOCK_CHECKPOINT_PATH
).to(DEVICE)

criterion_shape = nn.CrossEntropyLoss()
criterion_color = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE if 'LEARNING_RATE' in globals() else 1e-3
)

# Bỏ tham số verbose=True để tương thích với PyTorch mới trên Kaggle
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

print("✓ Mô hình MultiTaskResNet18 đã được khởi tạo thành công!")


In [ ]:
# ==========================================
# 7. VÒNG LẶP HUẤN LUYỆN DÀNH CHO HEADS (ĐẦY ĐỦ METRICS ACC & F1 CHO CẢ SHAPE & COLOR)
# ==========================================
history_heads = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_shape_acc": [],
    "val_shape_acc": [],
    "train_shape_f1": [],
    "val_shape_f1": [],
    "train_color_acc": [],
    "val_color_acc": [],
    "train_color_f1": [],
    "val_color_f1": [],
    "lr": [],
}

best_val_loss = float("inf")
start_time = time.time()

print("Bắt đầu Fine-tuning Heads với đầy đủ chỉ số (Accuracy & F1-Score)...")

for epoch in range(NUM_EPOCHS):
    # --- TRAIN PHASE ---
    model.train()
    model.backbone.eval()
    
    running_loss, total_samples = 0.0, 0
    train_shape_preds_list, train_shape_targets_list = [], []
    train_color_preds_list, train_color_targets_list = [], []

    for images, s_targets, c_targets in train_loader:
        images, s_targets, c_targets = (
            images.to(DEVICE),
            s_targets.to(DEVICE),
            c_targets.to(DEVICE),
        )
        optimizer.zero_grad()

        s_outputs, c_outputs = model(images)
        loss_shape = criterion_shape(s_outputs, s_targets)
        loss_color = criterion_color(c_outputs, c_targets)
        total_loss = loss_shape + COLOR_LOSS_WEIGHT * loss_color

        total_loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_samples += batch_size
        running_loss += total_loss.item() * batch_size

        _, s_preds = torch.max(s_outputs, 1)
        train_shape_preds_list.extend(s_preds.cpu().numpy())
        train_shape_targets_list.extend(s_targets.cpu().numpy())

        c_probs = torch.sigmoid(c_outputs)
        c_preds = (c_probs > 0.5).int()
        train_color_preds_list.append(c_preds.cpu().numpy())
        train_color_targets_list.append(c_targets.cpu().numpy())

    # Tính toán chỉ số Train
    epoch_train_loss = running_loss / total_samples
    tr_s_preds = np.array(train_shape_preds_list)
    tr_s_targets = np.array(train_shape_targets_list)
    tr_c_preds = np.vstack(train_color_preds_list)
    tr_c_targets = np.vstack(train_color_targets_list)

    epoch_train_shape_acc = float(np.mean(tr_s_preds == tr_s_targets))
    epoch_train_shape_f1 = float(f1_score(tr_s_targets, tr_s_preds, average="macro", zero_division=0))
    epoch_train_color_acc = float((tr_c_targets == tr_c_preds).mean(axis=0).mean())
    epoch_train_color_f1 = float(f1_score(tr_c_targets, tr_c_preds, average="macro", zero_division=0))

    # --- VALIDATION PHASE ---
    model.eval()
    val_running_loss, val_total_samples = 0.0, 0
    val_shape_preds_list, val_shape_targets_list = [], []
    val_color_preds_list, val_color_targets_list = [], []

    with torch.no_grad():
        for images, s_targets, c_targets in val_loader:
            images, s_targets, c_targets = (
                images.to(DEVICE),
                s_targets.to(DEVICE),
                c_targets.to(DEVICE),
            )

            s_outputs, c_outputs = model(images)
            loss_shape = criterion_shape(s_outputs, s_targets)
            loss_color = criterion_color(c_outputs, c_targets)
            total_loss = loss_shape + COLOR_LOSS_WEIGHT * loss_color

            batch_size = images.size(0)
            val_total_samples += batch_size
            val_running_loss += total_loss.item() * batch_size

            _, s_preds = torch.max(s_outputs, 1)
            val_shape_preds_list.extend(s_preds.cpu().numpy())
            val_shape_targets_list.extend(s_targets.cpu().numpy())

            c_probs = torch.sigmoid(c_outputs)
            c_preds = (c_probs > 0.5).int()
            val_color_preds_list.append(c_preds.cpu().numpy())
            val_color_targets_list.append(c_targets.cpu().numpy())

    # Tính toán chỉ số Val
    epoch_val_loss = val_running_loss / val_total_samples
    val_s_preds = np.array(val_shape_preds_list)
    val_s_targets = np.array(val_shape_targets_list)
    val_c_preds = np.vstack(val_color_preds_list)
    val_c_targets = np.vstack(val_color_targets_list)

    epoch_val_shape_acc = float(np.mean(val_s_preds == val_s_targets))
    epoch_val_shape_f1 = float(f1_score(val_s_targets, val_s_preds, average="macro", zero_division=0))
    epoch_val_color_acc = float((val_c_targets == val_c_preds).mean(axis=0).mean())
    epoch_val_color_f1 = float(f1_score(val_c_targets, val_c_preds, average="macro", zero_division=0))

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(epoch_val_loss)

    history_heads["epoch"].append(epoch + 1)
    history_heads["train_loss"].append(epoch_train_loss)
    history_heads["val_loss"].append(epoch_val_loss)
    history_heads["train_shape_acc"].append(epoch_train_shape_acc)
    history_heads["val_shape_acc"].append(epoch_val_shape_acc)
    history_heads["train_shape_f1"].append(epoch_train_shape_f1)
    history_heads["val_shape_f1"].append(epoch_val_shape_f1)
    history_heads["train_color_acc"].append(epoch_train_color_acc)
    history_heads["val_color_acc"].append(epoch_val_color_acc)
    history_heads["train_color_f1"].append(epoch_train_color_f1)
    history_heads["val_color_f1"].append(epoch_val_color_f1)
    history_heads["lr"].append(current_lr)

    log_msg = f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} | Loss: {epoch_train_loss:.4f}/{epoch_val_loss:.4f} | Shape Acc: {epoch_val_shape_acc:.4f} (F1: {epoch_val_shape_f1:.4f}) | Color Acc: {epoch_val_color_acc:.4f} (F1: {epoch_val_color_f1:.4f})"
    print(log_msg)
    logger_heads.info(log_msg)

    pd.DataFrame(history_heads).to_csv(PATHS["logs"] / "training_history.csv", index=False)

    # --- LƯU CHECKPOINT HEADS MỚI ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        print("  ✓ Val Loss cải thiện! Lưu checkpoint Heads mới...")
        ckpt_data = {
            "fc_shape": model.fc_shape.state_dict(),
            "fc_color": model.fc_color.state_dict(),
        }
        torch.save(ckpt_data, PATHS["checkpoints"] / "best_heads_finetuned.pth")
        torch.save(ckpt_data, Path("/kaggle/working/best_heads_finetuned.pth"))

final_ckpt = {
    "fc_shape": model.fc_shape.state_dict(),
    "fc_color": model.fc_color.state_dict(),
}
torch.save(final_ckpt, Path("/kaggle/working/best_heads_finetuned.pth"))
print("✓ Đã lưu checkpoint chính tại: /kaggle/working/best_heads_finetuned.pth")
print(f"\nHoàn thành Fine-tuning Heads trong {(time.time() - start_time)//60:.0f}m!")


In [ ]:
# ==========================================
# 8. VẼ ĐỒ THỊ HUẤN LUYỆN HEADS (6 SUBPLOTS)
# ==========================================
def save_learning_curves(history, output_plot_path, title_prefix="Stage 1 (Heads Fine-tuning)"):
    if not history or len(history.get("epoch", [])) == 0:
        print("⚠️ Chưa tìm thấy lịch sử huấn luyện.")
        return

    epochs = history.get("epoch", list(range(1, len(history.get("train_loss", [])) + 1)))

    plt.figure(figsize=(18, 10))

    # 1. Total Loss
    plt.subplot(2, 3, 1)
    if "train_loss" in history: plt.plot(epochs, history["train_loss"], label="Train Loss", color="blue", marker="o")
    if "val_loss" in history: plt.plot(epochs, history["val_loss"], label="Val Loss", color="red", marker="o")
    plt.title(f"{title_prefix} - Total Loss", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    # 2. Shape Accuracy
    plt.subplot(2, 3, 2)
    if "train_shape_acc" in history: plt.plot(epochs, history["train_shape_acc"], label="Train Shape Acc", color="blue", marker="o")
    if "val_shape_acc" in history: plt.plot(epochs, history["val_shape_acc"], label="Val Shape Acc", color="red", marker="o")
    plt.title(f"{title_prefix} - Shape Accuracy", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.legend()

    # 3. Color Accuracy
    plt.subplot(2, 3, 3)
    if "train_color_acc" in history: plt.plot(epochs, history["train_color_acc"], label="Train Color Acc", color="blue", marker="o")
    if "val_color_acc" in history: plt.plot(epochs, history["val_color_acc"], label="Val Color Acc", color="red", marker="o")
    plt.title(f"{title_prefix} - Color Accuracy (Macro)", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.legend()

    # 4. Shape F1
    plt.subplot(2, 3, 4)
    if "train_shape_f1" in history: plt.plot(epochs, history["train_shape_f1"], label="Train Shape F1", color="blue", marker="o")
    if "val_shape_f1" in history: plt.plot(epochs, history["val_shape_f1"], label="Val Shape F1", color="red", marker="o")
    plt.title(f"{title_prefix} - Shape F1 (Macro)", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.grid(True)
    plt.legend()

    # 5. Color F1
    plt.subplot(2, 3, 5)
    if "train_color_f1" in history: plt.plot(epochs, history["train_color_f1"], label="Train Color F1", color="blue", marker="o")
    if "val_color_f1" in history: plt.plot(epochs, history["val_color_f1"], label="Val Color F1", color="red", marker="o")
    plt.title(f"{title_prefix} - Color F1 (Macro)", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")
    plt.grid(True)
    plt.legend()

    # 6. Learning Rate
    plt.subplot(2, 3, 6)
    lr_vals = history.get("lr", [1e-4] * len(epochs))
    plt.plot(epochs, lr_vals, label="Learning Rate", color="green", marker="s")
    plt.title(f"{title_prefix} - Learning Rate", fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("LR")
    plt.yscale("log")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    output_plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_plot_path, dpi=300)
    plt.show()
    plt.close()

hist_data = globals().get("history_heads", {})
if not hist_data or len(hist_data.get("epoch", [])) == 0:
    csv_p = PATHS["logs"] / "training_history.csv"
    if csv_p.exists():
        df_hist = pd.read_csv(csv_p)
        hist_data = {col: df_hist[col].tolist() for col in df_hist.columns}

save_learning_curves(
    hist_data,
    PATHS["plots"] / "learning_curves.png",
    "Stage 1 (Heads Fine-tuning)"
)
print("Done plotting.")


In [ ]:
# ==========================================
# 9. ĐÁNH GIÁ TRÊN TEST SET & ĐẦY ĐỦ METRICS (STAGE 1)
# ==========================================
test_csv = COMBINED_DIR / "test_combined_crop.csv"
shape_encoder = getattr(train_dataset, 'shape_encoder', None)
mlb_color = getattr(train_dataset, 'mlb_color', None)

test_dataset = RxImageDataset(test_csv, IMG_DIR, transform=data_transforms["val"], shape_encoder=shape_encoder, mlb_color=mlb_color)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

ckpt_path = Path("/kaggle/working/best_heads_finetuned.pth")
if not ckpt_path.exists():
    ckpt_path = PATHS["checkpoints"] / "best_heads_finetuned.pth"

heads_weights = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
if isinstance(heads_weights, dict) and "fc_shape" in heads_weights:
    model.fc_shape.load_state_dict(heads_weights["fc_shape"])
    model.fc_color.load_state_dict(heads_weights["fc_color"])
else:
    model.load_state_dict(heads_weights, strict=False)
model.eval()

filenames = [str(f) for f in test_dataset.df.get("rxnavImageFileName", test_dataset.df.get("filename", [])).tolist()]
shape_preds, shape_logits_list, shape_targets = [], [], []
color_probs_list, color_targets_list = [], []

with torch.no_grad():
    for images, s_targets, c_targets in test_loader:
        images = images.to(DEVICE)
        s_outputs, c_outputs = model(images)

        _, s_preds = torch.max(s_outputs, 1)
        shape_preds.extend(s_preds.cpu().numpy())
        shape_logits_list.append(s_outputs.cpu().numpy())
        shape_targets.extend(s_targets.cpu().numpy())

        c_probs = torch.sigmoid(c_outputs)
        color_probs_list.append(c_probs.cpu().numpy())
        color_targets_list.append(c_targets.cpu().numpy())

shape_preds = np.array(shape_preds)
shape_logits = np.vstack(shape_logits_list)
shape_targets = np.array(shape_targets)
color_probs = np.vstack(color_probs_list)
color_targets = np.vstack(color_targets_list)

# --- THUẬT TOÁN TÌM NGƯỠNG TỐI ƯU CHO MÀU SẮC ---
best_thresholds = []
for i in range(NUM_COLOR_CLASSES):
    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.1, 0.9, 0.05):
        preds = (color_probs[:, i] > thresh).astype(int)
        score = f1_score(color_targets[:, i], preds, zero_division=0)
        if score > best_f1:
            best_f1, best_thresh = score, thresh
    best_thresholds.append(best_thresh)

best_thresholds = np.array(best_thresholds)
color_preds = (color_probs > best_thresholds).astype(int)

# DataFrame chi tiết dự đoán
pred_df = pd.DataFrame({
    "image_filename": filenames,
    "shape_target": shape_targets,
    "shape_pred": shape_preds,
})
color_names = test_dataset.color_cols
for i, col_name in enumerate(color_names):
    pred_df[f"target_{col_name}"] = color_targets[:, i]
    pred_df[f"pred_{col_name}"] = color_preds[:, i]
    pred_df[f"prob_{col_name}"] = color_probs[:, i]

# --- TÍNH TOÁN METRICS NÂNG CAO ---
all_shape_labels = np.arange(NUM_SHAPE_CLASSES)
if shape_encoder is not None and hasattr(shape_encoder, 'classes_'):
    shape_class_names = [str(cls) for cls in shape_encoder.classes_]
else:
    shape_class_names = [str(cls) for cls in all_shape_labels]

top3_shape_acc = (
    float(top_k_accuracy_score(shape_targets, shape_logits, k=3, labels=all_shape_labels))
    if NUM_SHAPE_CLASSES >= 3 else None
)

color_per_label_acc = (color_targets == color_preds).mean(axis=0)
color_mean_accuracy = float(np.mean(color_per_label_acc))
color_micro_accuracy = float((color_targets == color_preds).mean())
label_pos_counts = color_targets.sum(axis=0)
total_positives = label_pos_counts.sum()

color_weighted_accuracy = (
    float(np.sum(color_per_label_acc * label_pos_counts) / total_positives)
    if total_positives > 0 else color_mean_accuracy
)
color_exact_match_ratio = float(np.all(color_targets == color_preds, axis=1).mean())

metrics_summary = {
    "shape_accuracy": float(accuracy_score(shape_targets, shape_preds)),
    "shape_balanced_accuracy": float(balanced_accuracy_score(shape_targets, shape_preds)),
    "shape_top3_accuracy": top3_shape_acc,
    "shape_precision_macro": float(precision_score(shape_targets, shape_preds, average="macro", zero_division=0)),
    "shape_precision_weighted": float(precision_score(shape_targets, shape_preds, average="weighted", zero_division=0)),
    "shape_recall_macro": float(recall_score(shape_targets, shape_preds, average="macro", zero_division=0)),
    "shape_recall_weighted": float(recall_score(shape_targets, shape_preds, average="weighted", zero_division=0)),
    "shape_f1_macro": float(f1_score(shape_targets, shape_preds, average="macro", zero_division=0)),
    "shape_f1_weighted": float(f1_score(shape_targets, shape_preds, average="weighted", zero_division=0)),
    "shape_f1_micro": float(f1_score(shape_targets, shape_preds, average="micro", zero_division=0)),
    "color_macro_accuracy": color_mean_accuracy,
    "color_micro_accuracy": color_micro_accuracy,
    "color_weighted_accuracy": color_weighted_accuracy,
    "color_exact_match_ratio": color_exact_match_ratio,
    "color_f1_macro": float(f1_score(color_targets, color_preds, average="macro", zero_division=0)),
    "color_f1_micro": float(f1_score(color_targets, color_preds, average="micro", zero_division=0)),
    "color_f1_weighted": float(f1_score(color_targets, color_preds, average="weighted", zero_division=0)),
    "color_precision_macro": float(precision_score(color_targets, color_preds, average="macro", zero_division=0)),
    "color_precision_weighted": float(precision_score(color_targets, color_preds, average="weighted", zero_division=0)),
    "color_recall_macro": float(recall_score(color_targets, color_preds, average="macro", zero_division=0)),
    "color_recall_weighted": float(recall_score(color_targets, color_preds, average="weighted", zero_division=0)),
    "color_hamming_loss": float(hamming_loss(color_targets, color_preds)),
    "optimal_thresholds_used": {col: float(th) for col, th in zip(color_names, best_thresholds)}
}

shape_report = classification_report(shape_targets, shape_preds, labels=all_shape_labels, target_names=shape_class_names, output_dict=True, zero_division=0)
color_report = classification_report(color_targets, color_preds, target_names=color_names, output_dict=True, zero_division=0)

# Xuất file báo cáo & predictions
pred_df.to_csv(PATHS["predictions"] / "test_predictions_detailed.csv", index=False)
with open(PATHS["metrics"] / "overall_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=4)
with open(PATHS["metrics"] / "shape_classification_report.json", "w", encoding="utf-8") as f:
    json.dump(shape_report, f, indent=4)
with open(PATHS["metrics"] / "color_classification_report.json", "w", encoding="utf-8") as f:
    json.dump(color_report, f, indent=4)

# 10. Vẽ biểu đồ đồ thị
cm = confusion_matrix(shape_targets, shape_preds, labels=all_shape_labels)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=shape_class_names, yticklabels=shape_class_names)
plt.title("Confusion Matrix - Shape (Heads Finetuned)", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.savefig(PATHS["plots"] / "confusion_matrix_shape.png", dpi=300)
plt.show()
plt.close()

cm_norm = confusion_matrix(shape_targets, shape_preds, labels=all_shape_labels, normalize="true")
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=shape_class_names, yticklabels=shape_class_names)
plt.title("Normalized Confusion Matrix - Shape (Heads Finetuned)", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.savefig(PATHS["plots"] / "confusion_matrix_shape_normalized.png", dpi=300)
plt.show()
plt.close()

plt.figure(figsize=(12, 6))
bars = plt.bar(color_names, color_per_label_acc, color="skyblue", edgecolor="black")
plt.axhline(y=color_mean_accuracy, color="red", linestyle="--", label=f"Mean Acc: {color_mean_accuracy:.4f}")
plt.title("Per-Label Accuracy - Color (Multi-label)", fontsize=14, fontweight="bold")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.legend()
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 0.01, f"{yval:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(PATHS["plots"] / "color_per_label_accuracy.png", dpi=300)
plt.show()
plt.close()

print("\n=== HOÀN THÀNH! ĐÃ XUẤT ĐẦY ĐỦ CHECKPOINTS, LOGS, METRICS, PLOTS VÀ PREDICTIONS ===")
